### Notebook used for running simple benchmark

In [1]:
import utils as ut
import b_run_staging as b
import j_nb_benchmark as j
import polars as pl
import numpy as np

In [ ]:
user_counts = ut.load_data('user_counts', 'df')
user_mapping = ut.load_data('user_mapping', 'df')

train_test_dict = ut.load_json5('train_test_dict')

static_configs = ut.load_json5("static_configs")
runtime_configs = ut.load_json5("runtime_configs")
config_dict = ut.merge_configs(static_configs, runtime_configs)

# As currently using hurdle rather than nb benchmarks
config_dict['hurdle_model'] = True

bin_metric_dict = ut.load_json5("bin_metric_dict")
degen_mask = ut.load_data("degen_mask", "np")

config_nt_class, config_nt, train_test_nt_class, train_test_nt, bin_metric_nt = b.converting_dicts_to_nt(config_dict, train_test_dict, bin_metric_dict)

## Running the NB models in validation

In [ ]:
# Getting the NB mean and variance parameters for our static runner
usr_means, usr_vars, usr_p = j.get_user_hurdle_params(user_counts=user_counts, 
        n_users=user_mapping.shape[0], period_start=train_test_dict['train_start'], period_end=train_test_dict['burn_in_end'], config_dict=config_dict)
usr_coarse_means, usr_coarse_vars, usr_coarse_p = b.init_grid_hurdle(user_counts=user_counts.lazy(), n_users=user_mapping.shape[0], coarse_bins_per_day= bin_metric_dict['coarse_bins_per_day'], 
            period_start=train_test_dict['train_start'], period_end=train_test_dict['burn_in_end'], bin_metric_dict=bin_metric_dict)

AttributeError: 'DataFrame' object has no attribute 'collect'

In [4]:
batch_size = 5000

# Getting all validation bins and counts
valid_fine_bins = pl.DataFrame({'fine_bin_id': range(train_test_dict['validation_start'], train_test_dict['validation_end'])})
valid_counts = user_counts.filter((pl.col('fine_bin_id')>= train_test_dict['validation_start'])
                                  & (pl.col('fine_bin_id') < train_test_dict['validation_end'])).select('user_id', 'fine_bin_id', 'count')

# Init output metrics
valid_user_log_likelihood = 0
valid_user_hour_log_likelihood = 0
valid_n_scored = 0

for user_start in range(0, user_mapping.shape[0], batch_size):
    
    # Getting the users in the batch
    user_bc = user_mapping.select('user_id').slice(user_start, batch_size)
    bc_counts = valid_counts.filter(pl.col('user_id').is_in(user_bc['user_id']))

    # Creating a df with 0s filled in 
    all_valid_counts = user_bc.join(valid_fine_bins,how='cross').join(bc_counts, on=['user_id', 'fine_bin_id'] ,how='left').select(
        pl.col('user_id'), pl.col('fine_bin_id'), pl.col('count').fill_null(0)).to_numpy()

    # Running the hurdle benchmarks on that slice of users
    bc_user_log_likelihood, bc_user_hour_log_likelihood, bc_n_scored = j.run_hurdle_benchmarks(evaluation_counts=all_valid_counts, user_means=usr_means, user_variances=usr_vars, user_p=usr_p, 
        user_hour_means=usr_coarse_means, user_hour_variances=usr_coarse_vars, user_hour_p=usr_coarse_p, config_nt=config_nt, bin_metric_nt=bin_metric_nt, degen_mask=degen_mask)

    # Updating output metrics
    valid_user_log_likelihood += bc_user_log_likelihood
    valid_user_hour_log_likelihood += bc_user_hour_log_likelihood
    valid_n_scored += bc_n_scored

    del all_valid_counts
    del bc_counts

/tmp/ipykernel_237611/1453584398.py:17: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  bc_counts = valid_counts.filter(pl.col('user_id').is_in(user_bc['user_id']))


NameError: name 'usr_coarse_means' is not defined

In [ ]:
# Chaning results to a good format
valid_results = [{'model_name': 'user_hurdle', 'test_valid': 'valid', 'non_degen_ll': valid_user_log_likelihood / valid_n_scored},
                 {'model_name': 'user_hour_hurdle', 'test_valid': 'valid', 'non_degen_ll': valid_user_hour_log_likelihood / valid_n_scored}]

# Storing results
ut.store_run_results(results=valid_results, calibration_results=[], dir='benchmarks', run_name='benchmark_hurdle_valid')

### Running the hurdle models on test

In [ ]:
# Getting the hurdle mean and variance parameters for our static runner
usr_means, usr_vars, usr_p = j.get_user_hurdle_params(user_counts=user_counts, n_users=user_mapping.shape[0], 
        period_start=train_test_dict['train_start'], period_end=train_test_dict['validation_end'], config_dict=config_dict)

usr_coarse_means, usr_coarse_vars, usr_coarse_p = b.init_grid_hurdle(user_counts=user_counts.lazy(), n_users=user_mapping.shape[0], 
        coarse_bins_per_day=bin_metric_dict['coarse_bins_per_day'], period_start=train_test_dict['train_start'], period_end=train_test_dict['validation_end'], 
        bin_metric_dict=bin_metric_dict)

In [ ]:
batch_size = 5000

# Getting all testation bins and counts
test_fine_bins = pl.DataFrame({'fine_bin_id': range(train_test_dict['test_start'], train_test_dict['test_end'])})
test_counts = user_counts.filter((pl.col('fine_bin_id')>= train_test_dict['test_start'])
                                  & (pl.col('fine_bin_id') < train_test_dict['test_end'])).select('user_id', 'fine_bin_id', 'count')

# Init output metrics
test_user_log_likelihood = 0
test_user_hour_log_likelihood = 0
test_user_calibration = np.zeros(len(config_dict['calibration_thresholds']), dtype='float64')
test_user_hour_calibration = np.zeros(len(config_dict['calibration_thresholds']), dtype='float64')
test_n_scored = 0

for user_start in range(0, user_mapping.shape[0], batch_size):
    
    # Getting the users in the bactc
    user_bc = user_mapping.select('user_id').slice(user_start, batch_size)
    bc_counts = test_counts.filter(pl.col('user_id').is_in(user_bc['user_id']))

    # Creating a df with 0s filled in 
    all_test_counts = user_bc.join(test_fine_bins,how='cross').join(bc_counts, on=['user_id', 'fine_bin_id'] ,how='left').select(
        pl.col('user_id'), pl.col('fine_bin_id'), pl.col('count').fill_null(0)).to_numpy()
    
    # Running the nb benchmarks on that slice of users
    bc_user_log_likelihood, bc_user_hour_log_likelihood, bc_n_scored = j.run_hurdle_benchmarks(
                        evaluation_counts=all_test_counts, user_means=usr_means, user_variances=usr_vars, user_hour_means=usr_coarse_means, user_hour_variances=usr_coarse_vars, 
                        calibration_thresholds=config_dict['calibration_thresholds'],  config_nt=config_nt, bin_metric_nt=bin_metric_nt, degen_mask=degen_mask)

    # Updating output metrics
    test_user_log_likelihood += bc_user_log_likelihood
    test_user_hour_log_likelihood += bc_user_hour_log_likelihood
    test_n_scored += bc_n_scored

    # test_user_calibration += bc_user_calibration
    # test_user_hour_calibration += bc_user_hour_calibration
    del all_test_counts
    del bc_counts

In [ ]:
# Chaning results to a good format
test_results = [{'model_name': 'user_nb', 'test_valid': 'test', 'non_degen_ll': test_user_log_likelihood/ test_n_scored},
                 {'model_name': 'user_coarse_bin_nb', 'test_valid': 'test', 'non_degen_ll': test_user_hour_log_likelihood/ test_n_scored}]

test_calibration_results = []

for idx, threshold in enumerate(config_dict['calibration_thresholds']):
    test_calibration_results.append({'model_name': 'user_nb', 'test_valid': 'test', 'threshold': threshold, 
                                      'calibration': test_user_calibration[idx]/ test_n_scored})
    # test_calibration_results.append({'model_name': 'user_coarse_bin_nb', 'test_valid': 'test', 'threshold': threshold, 
    #                                   'calibration': test_user_hour_calibration[idx]/ test_n_scored})
    
#Storing results
ut.store_run_results(results=test_results, calibration_results=test_calibration_results, dir="benchmarks", run_name="benchmark_hurdle_test")